## Instruções para Execução do Projeto

Antes de executar o código no Google Colab, é necessário realizar o upload dos arquivos utilizados durante o desenvolvimento e a análise dos resultados.

Os seguintes arquivos devem estar disponíveis no ambiente do Colab:

* **pokemon_processed.csv** → Dataset pré-processado utilizado pelos algoritmos de classificação.
* **metricas_finais.csv** → Arquivo contendo as métricas finais obtidas para os melhores modelos.
* **resultados_testes_parametros.csv** → Arquivo contendo todos os testes de parâmetros realizados pelo GridSearchCV.

### Como realizar o upload

1. Abrir o Google Colab.
2. Selecionar o ícone de pasta localizado na barra lateral esquerda.
3. Clicar em **Upload para sessão de armazenamento**.
4. Selecionar os três arquivos:

   * `pokemon_processed.csv`
   * `metricas_finais.csv`
   * `resultados_testes_parametros.csv`
5. Verificar se os arquivos aparecem na área de arquivos do Colab antes de executar o notebook.

Após o upload dos arquivos, o código poderá ser executado normalmente, permitindo a reprodução dos experimentos e a visualização dos resultados obtidos.


# 1. Importações

In [35]:
# Biblioteca utilizada para manipulação e análise dos dados em formato de tabela
import pandas as pd


# Algoritmos de classificação utilizados no projeto
# Cada um será treinado e comparado para prever a classe "legendary"
from sklearn.tree import DecisionTreeClassifier          # Árvore de Decisão
from sklearn.neighbors import KNeighborsClassifier       # K-Nearest Neighbors (KNN)
from sklearn.naive_bayes import GaussianNB               # Naive Bayes
from sklearn.linear_model import LogisticRegression      # Regressão Logística
from sklearn.svm import SVC                              # Support Vector Machine (SVM)
from sklearn.neural_network import MLPClassifier         # Rede Neural MLP


# Ferramentas para realizar a validação cruzada e ajuste dos parâmetros
from sklearn.model_selection import StratifiedKFold      # Validação cruzada estratificada 10-fold
from sklearn.model_selection import GridSearchCV         # Busca das melhores combinações de parâmetros

# 2. Carregando o dataset

In [36]:
# Carregamento do dataset pré-processado
# O arquivo pokemon_processed.csv contém os dados já pré-processados.
# A função read_csv() realiza a leitura do arquivo e armazena os dados em uma tabela (DataFrame).
# Para a implementação desse código foi feito o upload do arquivo pokemon_processed.csv para
# o local /content/pokemon_processed.csv no Google Colab.

df = pd.read_csv("/content/pokemon_processed.csv")

## 2.1 Exibindo as primeiras linhas

In [37]:
# Exibindo as primeiras linhas da base para verificar se os dados foram carregados corretamente

print(df.head())

      total        hp    attack   defense  sp_attack  sp_defense     speed  \
0  0.150526  0.173228  0.237838  0.179592   0.298913    0.195652  0.205128   
1  0.242105  0.232283  0.308108  0.236735   0.380435    0.260870  0.282051   
2  0.368421  0.311024  0.416216  0.318367   0.489130    0.347826  0.384615   
3  0.473684  0.311024  0.513514  0.481633   0.608696    0.434783  0.384615   
4  0.368421  0.311024  0.416216  0.318367   0.489130    0.347826  0.384615   

   generation  legendary  type1_Blastoise  ...  type2_Grass  type2_Ground  \
0           1          0                0  ...            0             0   
1           1          0                0  ...            0             0   
2           1          0                0  ...            0             0   
3           1          0                0  ...            0             0   
4           1          0                0  ...            0             0   

   type2_Ice  type2_None  type2_Normal  type2_Poison  type2_Psychic 

## 2.2 Informações gerais do dataset

In [38]:
# Apresentando informações gerais do dataset:
# quantidade de registros, tipos das variáveis e possíveis valores ausentes.

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072 entries, 0 to 1071
Data columns (total 48 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   total            1072 non-null   float64
 1   hp               1072 non-null   float64
 2   attack           1072 non-null   float64
 3   defense          1072 non-null   float64
 4   sp_attack        1072 non-null   float64
 5   sp_defense       1072 non-null   float64
 6   speed            1072 non-null   float64
 7   generation       1072 non-null   int64  
 8   legendary        1072 non-null   int64  
 9   type1_Blastoise  1072 non-null   int64  
 10  type1_Bug        1072 non-null   int64  
 11  type1_Dark       1072 non-null   int64  
 12  type1_Dragon     1072 non-null   int64  
 13  type1_Electric   1072 non-null   int64  
 14  type1_Fairy      1072 non-null   int64  
 15  type1_Fighting   1072 non-null   int64  
 16  type1_Fire       1072 non-null   int64  
 17  type1_Flying  

# 3. Separando X e y

In [39]:
# Separação das variáveis de entrada (X) e da variável alvo (y)

# X contém os atributos utilizados pelos algoritmos de classificação,
# como HP, ataque, defesa, velocidade, geração, entre outros.

# y representa a classe que será prevista pelo modelo,
# indicando se o Pokémon é lendário ou não (coluna "legendary").

X = df.drop("legendary", axis=1)

y = df["legendary"]

In [40]:
# Visualização dos atributos utilizados como entrada

X.head()

,total,hp,attack,defense,sp_attack,sp_defense,speed,generation,type1_Blastoise,type1_Bug,...,type2_Grass,type2_Ground,type2_Ice,type2_None,type2_Normal,type2_Poison,type2_Psychic,type2_Rock,type2_Steel,type2_Water
0,0.150526,0.173228,0.237838,0.179592,0.298913,0.195652,0.205128,1,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0.242105,0.232283,0.308108,0.236735,0.380435,0.260870,0.282051,1,0,0,...,0,0,0,0,0,1,0,0,0,0
2,0.368421,0.311024,0.416216,0.318367,0.489130,0.347826,0.384615,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0.473684,0.311024,0.513514,0.481633,0.608696,0.434783,0.384615,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0.368421,0.311024,0.416216,0.318367,0.489130,0.347826,0.384615,1,0,0,...,0,0,0,0,0,1,0,0,0,0


In [41]:
# Visualização da classe que será prevista

y.head()

,legendary
0,0
1,0
2,0
3,0
4,0


# 4. Efetuando a validação cruzada

In [42]:
# Configuração da validação cruzada estratificada (10-fold)

# O método StratifiedKFold divide o dataset em 10 partes (folds),
# mantendo a mesma proporção das classes da variável alvo (legendary)
# em cada divisão.

# Em cada etapa, uma parte dos dados é utilizada para teste e as demais
# para treinamento do modelo. Esse processo é repetido 10 vezes e,
# ao final, os resultados são utilizados para calcular a média do desempenho.

from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(

    n_splits=10,        # Número de divisões da validação cruzada

    shuffle=True,       # Mistura os dados antes da divisão

    random_state=42     # Garante que o resultado possa ser reproduzido

)

print("Número de folds:", cv.get_n_splits())

Número de folds: 10


# 5. Criando os modelos

In [43]:
# Instanciação dos algoritmos de classificação utilizados no projeto

# Nesta etapa são criados os modelos que serão avaliados e comparados
# utilizando validação cruzada estratificada 10-fold.

# Os algoritmos testados serão:
# - Árvore de Decisão
# - K-Nearest Neighbors (KNN)
# - Naive Bayes
# - Regressão Logística
# - Support Vector Machine (SVM)
# - Rede Neural MLP

# Cada modelo será posteriormente submetido ao ajuste de parâmetros
# e terá seu desempenho avaliado por meio das métricas de classificação.

modelos = {

    "Árvore": DecisionTreeClassifier(),

    "KNN": KNeighborsClassifier(),

    "Naive Bayes": GaussianNB(),

    "Logística": LogisticRegression(max_iter=1000),

    "SVM": SVC(),

    "MLP": MLPClassifier(max_iter=1000)

}

# 6. Definindo os parâmetros

In [44]:
# Definição dos parâmetros que serão testados para cada algoritmo

# O GridSearchCV utilizará esses valores para testar diferentes
# configurações dos modelos e identificar aquela que apresenta
# o melhor desempenho durante a validação cruzada 10-fold.

parametros = {

    # Parâmetros da Árvore de Decisão
    # max_depth controla a profundidade máxima da árvore.
    # Valores maiores permitem regras mais complexas.
    "Árvore": {

        "max_depth": [3, 5, 10]

    },

    # Parâmetros do KNN
    # n_neighbors define a quantidade de vizinhos mais próximos
    # utilizada para classificar uma nova amostra.
    "KNN": {

        "n_neighbors": [3, 5, 7]

    },

    # Parâmetros do Naive Bayes
    # O parâmetro var_smoothing adiciona um pequeno valor à variância
    # dos atributos para evitar problemas numéricos durante os cálculos
    # de probabilidade realizados pelo algoritmo.
    "Naive Bayes": {
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },

    # Parâmetros da Regressão Logística
    # C controla o nível de regularização do modelo.
    # Valores menores aplicam maior regularização.
    "Logística": {

        "C": [0.1, 1, 10]

    },

    # Parâmetros do SVM
    # C controla a penalização dos erros de classificação.
    # Diferentes valores podem alterar a capacidade de generalização.
    "SVM": {

        "C": [0.1, 1, 10]

    },

    # Parâmetros da Rede Neural MLP
    # hidden_layer_sizes define a arquitetura da rede neural,
    # indicando a quantidade de neurônios nas camadas ocultas.
    "MLP": {

        "hidden_layer_sizes": [

            (50,),      # 1 camada oculta com 50 neurônios

            (100,),     # 1 camada oculta com 100 neurônios

            (50, 50)    # 2 camadas ocultas com 50 neurônios cada

        ]

    }

}

# 7. Métricas utilizadas na avaliação dos modelos

In [45]:
# As métricas selecionadas para esse projeto foram:

# Acurácia (Accuracy): proporção total de classificações corretas.
# Precisão (Precision): proporção de previsões positivas que realmente pertencem à classe positiva.
# Recall: capacidade do modelo de identificar corretamente os exemplos da classe positiva.
# F1-score: média harmônica entre Precision e Recall, fornecendo uma medida equilibrada.

# O make_scorer é utilizado para adaptar as métricas ao formato
# exigido pelo GridSearchCV.

# Importação das métricas utilizadas na avaliação dos modelos
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

# Definição das métricas que serão calculadas durante a validação cruzada
metricas = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}

# 8. Função para testar os modelos

In [46]:
# Função responsável por treinar, ajustar e avaliar cada algoritmo

# Para cada modelo de classificação, esta função realiza:

# 1. Busca dos melhores hiperparâmetros utilizando GridSearchCV;
# 2. Avaliação do modelo por meio de validação cruzada estratificada 10-fold;
# 3. Cálculo das métricas de desempenho:
#    - Acurácia (Accuracy)
#    - Precisão (Precision)
#    - Recall
#    - F1-score
# 4. Identificação da melhor combinação de parâmetros;
# 5. Armazenamento dos resultados para posterior comparação entre os algoritmos.

# Ao final da execução são retornados:
# - As métricas do melhor modelo encontrado;
# - Todos os resultados gerados pelo GridSearchCV.

def testar_modelo(nome, modelo, parametro):

    # Configuração do GridSearchCV

    # O GridSearchCV testa automaticamente todas as combinações
    # de parâmetros definidas para o algoritmo.

    # A avaliação é realizada utilizando validação cruzada
    # estratificada 10-fold.

    # O parâmetro refit="accuracy" determina que o melhor modelo
    # será escolhido com base na acurácia média obtida.

    grid = GridSearchCV(
        modelo,
        parametro,
        cv=cv,
        scoring=metricas,
        refit="accuracy",
        return_train_score=False
    )

    # Execução do treinamento e avaliação do modelo
    # utilizando todas as divisões da validação cruzada.

    grid.fit(X, y)

    # Exibição das principais informações do melhor modelo.

    print("\nModelo:", nome)
    print("Melhor parâmetro:", grid.best_params_)
    print("Melhor accuracy:", grid.best_score_)


    # Armazena todos os resultados produzidos pelo GridSearchCV.

    # Essa tabela contém o desempenho de cada combinação
    # de parâmetros testada durante os experimentos.

    testes_parametros = pd.DataFrame(grid.cv_results_)
    testes_parametros["Modelo"] = nome


    # Identifica a posição do melhor conjunto de parâmetros.

    melhor_indice = grid.best_index_

    # Armazena as métricas médias e os desvios padrão
    # obtidos pelo melhor modelo durante a validação cruzada.

    # As médias representam o desempenho geral do algoritmo.
    # Os desvios padrão indicam a estabilidade dos resultados
    # entre os diferentes folds.

    resultado_modelo = {
        "Modelo": nome,
        "Melhores Parâmetros": grid.best_params_,
        "Accuracy Média": grid.cv_results_["mean_test_accuracy"][melhor_indice],
        "Accuracy Desvio": grid.cv_results_["std_test_accuracy"][melhor_indice],
        "Precision Média": grid.cv_results_["mean_test_precision"][melhor_indice],
        "Precision Desvio": grid.cv_results_["std_test_precision"][melhor_indice],
        "Recall Média": grid.cv_results_["mean_test_recall"][melhor_indice],
        "Recall Desvio": grid.cv_results_["std_test_recall"][melhor_indice],
        "F1 Média": grid.cv_results_["mean_test_f1"][melhor_indice],
        "F1 Desvio": grid.cv_results_["std_test_f1"][melhor_indice]
    }

    # Retorna:
    # 1. Resumo das métricas do melhor modelo;
    # 2. Tabela completa com todos os testes realizados.

    return resultado_modelo, testes_parametros

# 9. Executando todos os modelos

In [47]:
# Estruturas utilizadas para armazenar os resultados dos experimentos

# resultados_finais:
# Armazena um resumo do melhor desempenho obtido por cada algoritmo,
# incluindo os melhores parâmetros, médias e desvios padrão das métricas.

# todos_testes:
# Armazena os resultados completos produzidos pelo GridSearchCV,
# contendo todas as combinações de parâmetros avaliadas durante os testes.

resultados_finais = []
todos_testes = []

# Execução dos algoritmos de classificação

# Nesta etapa, cada algoritmo definido anteriormente é submetido
# ao processo de ajuste de parâmetros e avaliação por validação
# cruzada estratificada 10-fold.

# Para cada modelo são obtidos:
# - Melhor conjunto de parâmetros;
# - Acurácia (Accuracy) média e desvio padrão;
# - Precisão (Precision) média e desvio padrão;
# - Recall média e desvio padrão;
# - F1-score médio e desvio padrão.

# Os resultados são armazenados para posterior comparação entre
# os algoritmos e elaboração das tabelas e gráficos do artigo.

for nome in modelos:

    # Executa os testes do algoritmo atual
    resultado_modelo, testes_parametros = testar_modelo(
        nome,
        modelos[nome],
        parametros[nome]
    )

    # Armazena o resumo do melhor resultado obtido pelo algoritmo
    resultados_finais.append(resultado_modelo)

    # Armazena todos os testes de parâmetros realizados
    todos_testes.append(testes_parametros)


Modelo: Árvore
Melhor parâmetro: {'max_depth': 5}
Melhor accuracy: 0.9552180685358256

Modelo: KNN
Melhor parâmetro: {'n_neighbors': 3}
Melhor accuracy: 0.9300276912426446

Modelo: Naive Bayes
Melhor parâmetro: {'var_smoothing': 1e-07}
Melhor accuracy: 0.2947040498442367

Modelo: Logística
Melhor parâmetro: {'C': 10}
Melhor accuracy: 0.9374956732433368

Modelo: SVM
Melhor parâmetro: {'C': 10}
Melhor accuracy: 0.9449723087573554

Modelo: MLP
Melhor parâmetro: {'hidden_layer_sizes': (100,)}
Melhor accuracy: 0.939373485635168


# 10. Resultados finais

In [48]:
# Organização e armazenamento dos resultados finais

# Após a execução de todos os algoritmos, os resultados obtidos
# são organizados em tabelas para facilitar a análise e a comparação
# do desempenho dos modelos.

# Essas tabelas serão utilizadas posteriormente na elaboração
# das tabelas, gráficos e discussões presentes no artigo científico.

# Criação da tabela contendo apenas os melhores resultados
# encontrados para cada algoritmo.

# Para cada modelo são armazenados:
# - Melhores parâmetros encontrados;
# - Acurácia (Accuracy) média e desvio padrão;
# - Precisão (Precision) média e desvio padrão;
# - Recall média e desvio padrão;
# - F1-score médio e desvio padrão.

tabela_metricas = pd.DataFrame(resultados_finais)

# Criação da tabela contendo todos os experimentos realizados.

# Esta tabela reúne todas as combinações de parâmetros avaliadas
# pelo GridSearchCV durante os testes dos algoritmos.

# Ela permite analisar o impacto dos diferentes parâmetros
# sobre o desempenho dos modelos.

tabela_testes = pd.concat(todos_testes, ignore_index=True)

# Salvamento dos resultados em arquivos CSV.

# Esses arquivos podem ser compartilhados com os demais integrantes
# do grupo e utilizados como base para a construção das tabelas
# e gráficos do artigo.

# Arquivo contendo apenas os melhores resultados de cada algoritmo.
tabela_metricas.to_csv("metricas_finais.csv", index=False)

# Arquivo contendo todos os testes de parâmetros realizados.
tabela_testes.to_csv("resultados_testes_parametros.csv", index=False)

# Exibe na tela o resumo final dos resultados obtidos.
print(tabela_metricas)

        Modelo             Melhores Parâmetros  Accuracy Média  \
0       Árvore                {'max_depth': 5}        0.955218   
1          KNN              {'n_neighbors': 3}        0.930028   
2  Naive Bayes        {'var_smoothing': 1e-07}        0.294704   
3    Logística                       {'C': 10}        0.937496   
4          SVM                       {'C': 10}        0.944972   
5          MLP  {'hidden_layer_sizes': (100,)}        0.939373   

   Accuracy Desvio  Precision Média  Precision Desvio  Recall Média  \
0         0.010085         0.809802          0.054053      0.779545   
1         0.019745         0.829762          0.112129      0.447727   
2         0.057029         0.130629          0.013490      0.949242   
3         0.015047         0.821538          0.142268      0.591667   
4         0.012096         0.856605          0.118032      0.617424   
5         0.008535         0.820260          0.097355      0.609091   

   Recall Desvio  F1 Média  F1 Desvio  

# 10.1 Metricas finais

In [49]:
import pandas as pd

df = pd.read_csv("metricas_finais.csv")

df

,Modelo,Melhores Parâmetros,Accuracy Média,Accuracy Desvio,Precision Média,Precision Desvio,Recall Média,Recall Desvio,F1 Média,F1 Desvio
0,Árvore,{'max_depth': 5},0.955218,0.010085,0.809802,0.054053,0.779545,0.093120,0.790863,0.052031
1,KNN,{'n_neighbors': 3},0.930028,0.019745,0.829762,0.112129,0.447727,0.163343,0.569245,0.156668
2,Naive Bayes,{'var_smoothing': 1e-07},0.294704,0.057029,0.130629,0.013490,0.949242,0.055778,0.229486,0.021709
3,Logística,{'C': 10},0.937496,0.015047,0.821538,0.142268,0.591667,0.103733,0.674168,0.071485
4,SVM,{'C': 10},0.944972,0.012096,0.856605,0.118032,0.617424,0.082241,0.710724,0.064264
5,MLP,"{'hidden_layer_sizes': (100,)}",0.939373,0.008535,0.820260,0.097355,0.609091,0.127633,0.681806,0.070904


# 10.2 Resultados dos testes de parametros

In [50]:
import pandas as pd

testes = pd.read_csv("resultados_testes_parametros.csv")

display(testes.head(20))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,...,split8_test_f1,split9_test_f1,mean_test_f1,std_test_f1,rank_test_f1,Modelo,param_n_neighbors,param_var_smoothing,param_C,param_hidden_layer_sizes
0,0.008154,0.000556,0.015590,0.001124,3.0,{'max_depth': 3},0.953704,0.925926,0.915888,0.962617,...,0.782609,0.695652,0.732053,0.083272,3,Árvore,NaN,NaN,NaN,NaN
1,0.011233,0.001860,0.017981,0.001779,5.0,{'max_depth': 5},0.962963,0.953704,0.953271,0.953271,...,0.818182,0.727273,0.790863,0.052031,1,Árvore,NaN,NaN,NaN,NaN
2,0.008579,0.001078,0.010080,0.001695,10.0,{'max_depth': 10},0.953704,0.935185,0.934579,0.953271,...,0.800000,0.800000,0.776328,0.069313,2,Árvore,NaN,NaN,NaN,NaN
3,0.002860,0.000278,0.011847,0.000270,NaN,{'n_neighbors': 3},0.935185,0.935185,0.925234,0.925234,...,0.500000,0.818182,0.569245,0.156668,1,KNN,3.0,NaN,NaN,NaN
4,0.004309,0.001119,0.017023,0.003413,NaN,{'n_neighbors': 5},0.907407,0.888889,0.906542,0.925234,...,0.375000,0.800000,0.428962,0.175377,2,KNN,5.0,NaN,NaN,NaN
5,0.004533,0.000362,0.020283,0.001328,NaN,{'n_neighbors': 7},0.907407,0.898148,0.906542,0.906542,...,0.400000,0.500000,0.338753,0.102512,3,KNN,7.0,NaN,NaN,NaN
6,0.004711,0.001063,0.012577,0.002921,NaN,{'var_smoothing': 1e-09},0.342593,0.203704,0.233645,0.252336,...,0.190476,0.226415,0.219977,0.017316,3,Naive Bayes,NaN,1.000000e-09,NaN,NaN
7,0.003716,0.000442,0.009800,0.001159,NaN,{'var_smoothing': 1e-08},0.351852,0.222222,0.252336,0.261682,...,0.194175,0.228571,0.221841,0.016704,2,Naive Bayes,NaN,1.000000e-08,NaN,NaN
8,0.003661,0.000450,0.009462,0.000647,NaN,{'var_smoothing': 1e-07},0.407407,0.259259,0.261682,0.299065,...,0.194175,0.233010,0.229486,0.021709,1,Naive Bayes,NaN,1.000000e-07,NaN,NaN
9,0.011532,0.001479,0.012874,0.000450,NaN,{'C': 0.1},0.898148,0.907407,0.906542,0.897196,...,0.000000,0.000000,0.133150,0.119467,3,Logística,NaN,NaN,0.1,NaN
